In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# git clone
!git clone --depth 1 https://github.com/TadasBaltrusaitis/OpenFace

%cd /kaggle/working/OpenFace 

Cloning into 'OpenFace'...
remote: Enumerating objects: 3784, done.
remote: Counting objects: 100% (3784/3784), done.
remote: Compressing objects: 100% (3053/3053), done.
remote: Total 3784 (delta 709), reused 3336 (delta 686), pack-reused 0 (from 0)
Receiving objects: 100% (3784/3784), 519.28 MiB | 24.22 MiB/s, done.
Resolving deltas: 100% (709/709), done.
Updating files: 100% (3672/3672), done.
/kaggle/working/OpenFace


In [3]:
# 의존성 설치
import subprocess, sys

def run(cmd, check=True):
    result = subprocess.run(cmd, shell=True,
                            stdout=sys.stdout, stderr=sys.stderr, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f"실패: {cmd}")

run("apt-get update -qq")
run("apt-get install -y -q --no-install-recommends "
    "build-essential cmake zip "
    "libopenblas-dev liblapack-dev "
    "libgtk2.0-dev pkg-config "
    "libavcodec-dev libavformat-dev "
    "libtbb-dev libjpeg-dev libpng-dev libtiff-dev libdc1394-dev")

run("pip install -q mediapipe retina-face")  

print("✅ 의존성 설치 완료")

✅ 의존성 설치 완료


In [4]:
# OpenCV 4.1.0 build
import os, multiprocessing
from pathlib import Path

os.chdir("/kaggle/working")
run("wget -q https://github.com/opencv/opencv/archive/4.1.0.zip")
run("unzip -q 4.1.0.zip")
Path("/kaggle/working/opencv-4.1.0/build").mkdir(parents=True)
os.chdir("/kaggle/working/opencv-4.1.0/build")

run("cmake "
    "-D CMAKE_BUILD_TYPE=RELEASE "
    "-D CMAKE_INSTALL_PREFIX=/usr/local "
    "-D WITH_TBB=OFF "
    "-D WITH_OPENMP=OFF "
    "-D WITH_CUDA=OFF "
    "-D BUILD_SHARED_LIBS=OFF "
    "-D BUILD_TESTS=OFF "
    "-D BUILD_PERF_TESTS=OFF "
    "-D BUILD_EXAMPLES=OFF "
    "-D OpenBLAS_INCLUDE_DIR=/usr/include/x86_64-linux-gnu "
    "-D OpenBLAS_LIB=/usr/lib/x86_64-linux-gnu/libopenblas.so "
    "..")

run("make -j2")
run("make install")
run("ldconfig")
os.chdir("/kaggle/working")
run("rm -rf opencv-4.1.0 4.1.0.zip")
print("✅ OpenCV 설치 완료")

✅ OpenCV 설치 완료


In [5]:
# dlib 19.13 build
os.chdir("/kaggle/working")
run("wget -q http://dlib.net/files/dlib-19.13.tar.bz2")
run("tar xf dlib-19.13.tar.bz2")
os.chdir("/kaggle/working/dlib-19.13")
Path("build").mkdir(exist_ok=True)
os.chdir("build")
run("cmake ..")
run(f"cmake --build . --config Release -- -j2")
run("make install")
run("ldconfig")
os.chdir("/kaggle/working")
run("rm -rf dlib-19.13 dlib-19.13.tar.bz2")
print("✅ dlib 설치 완료")

✅ dlib 설치 완료


In [6]:
# OpenFace 모델 다운로드
os.chdir("/kaggle/working/OpenFace")  # ← OpenFace 폴더 안으로 이동
run("bash download_models.sh")
print("✅ 모델 다운로드 완료")

✅ 모델 다운로드 완료


In [7]:
# OpenFace build
run("rm -rf /kaggle/working/OpenFace/build")
Path("/kaggle/working/OpenFace/build").mkdir()
os.chdir("/kaggle/working/OpenFace/build")

run("cmake -D CMAKE_BUILD_TYPE=RELEASE -D CMAKE_CXX_FLAGS='-std=c++17' ..")
run("make -j2")
os.chdir("/kaggle/working")

BIN = Path("/kaggle/working/OpenFace/build/bin/FeatureExtraction")
print("✅ OpenFace 빌드 완료!" if BIN.exists() else "❌ 빌드 실패")

✅ OpenFace 빌드 완료!


In [10]:
# 모델 파일 복사 (patch_experts → bin/model/)
import shutil

SRC_PATCH = Path("/kaggle/working/OpenFace/lib/local/LandmarkDetector/model/patch_experts")
DST_PATCH = Path("/kaggle/working/OpenFace/build/bin/model/patch_experts")
DST_PATCH.mkdir(parents=True, exist_ok=True)

for f in SRC_PATCH.glob("*.dat"):
    dst = DST_PATCH / f.name
    if not dst.exists():
        shutil.copy2(f, dst)
        print(f"복사: {f.name}")

print("✅ 모델 파일 복사 완료")
print("파일 목록:", [f.name for f in DST_PATCH.glob("*.dat")])

✅ 모델 파일 복사 완료
파일 목록: ['cen_patches_0.35_of.dat', 'cen_patches_1.00_of.dat', 'cen_patches_0.25_of.dat', 'cen_patches_0.50_of.dat']


In [11]:
# 메타데이터 구축
from tqdm import tqdm

# ── 데이터셋 경로 설정  ────────────────────────
DFD_ROOT   = Path("/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset")
CELEB_ROOT = Path("/kaggle/input/datasets/reubensuju/celeb-df-v2")

# ── 영상 목록 + 라벨 구성 ────────────────────────────────────────
video_list = []

# DFD
for mp4 in DFD_ROOT.rglob("*.mp4"):
    path_str = str(mp4).lower()
    label = 0 if "original" in path_str else 1
    video_list.append({"path": str(mp4), "label": label, "source": "DFD"})

# Celeb-DF
for mp4 in CELEB_ROOT.rglob("*.mp4"):
    label = 1 if "synthesis" in str(mp4).lower() else 0
    video_list.append({"path": str(mp4), "label": label, "source": "CelebDF"})

print(f"전체 영상 수: {len(video_list)}")
print(f"  real: {sum(1 for v in video_list if v['label']==0)}")
print(f"  fake: {sum(1 for v in video_list if v['label']==1)}")

전체 영상 수: 9960
  real: 4321
  fake: 5639


In [12]:
# AU 추출 함수 정의
import subprocess, shutil, tempfile
import pandas as pd
from pathlib import Path

BIN = "/kaggle/working/OpenFace/build/bin/FeatureExtraction"

AU_COLS = [
    "AU01_r", "AU02_r", "AU04_r", "AU05_r", "AU06_r", "AU07_r",
    "AU09_r", "AU10_r", "AU12_r", "AU14_r", "AU15_r", "AU17_r",
    "AU20_r", "AU23_r", "AU24_r", "AU25_r", "AU26_r",
]

def extract_au(video_path: str) -> pd.DataFrame | None:
    """
    영상 경로 → AU 수치 DataFrame 반환.
    - 경로 공백 자동 처리 (임시 경로로 복사)
    - mp4 → avi 변환 (OpenFace mp4 코덱 미지원 대응)
    - confidence < 0.85 프레임 제거
    - 실패 시 None 반환
    """
    with tempfile.TemporaryDirectory(dir="/kaggle/working") as tmpdir:
        tmpdir = Path(tmpdir)

        # 공백 없는 경로로 복사
        tmp_mp4 = tmpdir / "input.mp4"
        shutil.copy2(video_path, tmp_mp4)

        # mp4 → avi 변환
        tmp_avi = tmpdir / "input.avi"
        ret = subprocess.call(
            f"ffmpeg -y -loglevel error "
            f"-i '{tmp_mp4}' -vcodec mjpeg -q:v 3 '{tmp_avi}'",
            shell=True
        )
        if ret != 0 or not tmp_avi.exists():
            return None

        # OpenFace 실행
        out_dir = tmpdir / "out"
        out_dir.mkdir()
        ret = subprocess.call(
            f'"{BIN}" -f "{tmp_avi}" -out_dir "{out_dir}" -aus -q',
            shell=True
        )

        # CSV 파싱
        csv_files = list(out_dir.glob("*.csv"))
        if not csv_files:
            return None

        df = pd.read_csv(csv_files[0])
        df.columns = df.columns.str.strip()

        # confidence 필터링
        df = df[df["confidence"] >= 0.85].reset_index(drop=True)
        if len(df) < 2:
            return None

        available = [c for c in AU_COLS if c in df.columns]
        return df[["frame", "timestamp", "confidence"] + available]

In [14]:
# 상태 확인
from pathlib import Path

print(f"video_list 길이: {len(video_list)}")
print(f"첫 번째 영상: {video_list[0] if video_list else '없음'}")

# OUTPUT_DIR 확인
OUTPUT_DIR = Path("/kaggle/working/processed")
print(f"\nOUTPUT_DIR 존재: {OUTPUT_DIR.exists()}")
if OUTPUT_DIR.exists():
    existing = list(OUTPUT_DIR.glob("test_*"))
    print(f"기존 test_ 폴더 수: {len(existing)}")
    for d in existing:
        print(f"  {d.name}")
        for f in d.rglob("*"):
            print(f"    {f.name}")

video_list 길이: 9960
첫 번째 영상: {'path': '/kaggle/input/datasets/sanikatiwarekar/deep-fake-detection-dfd-entire-original-dataset/DFD_original sequences/26__walking_down_street_outside_angry.mp4', 'label': 0, 'source': 'DFD'}

OUTPUT_DIR 존재: False


In [109]:
import shutil
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/processed")

# test_ 로 시작하는 폴더 전부 삭제
for d in OUTPUT_DIR.glob("test_*"):
    shutil.rmtree(d)
    print(f"삭제: {d.name}")

print("✅ 테스트 결과물 정리 완료")

삭제: test_004_05__outside_talking_still_laughing
삭제: test_003_08__walking_down_street_outside_angry
삭제: test_000_26__walking_down_street_outside_angry
삭제: test_002_14__walking_down_indoor_hall_disgust
삭제: test_001_08__talking_against_wall
✅ 테스트 결과물 정리 완료


In [114]:
# 전처리 파이프라인 전체 코드
import cv2
import json
import numpy as np
import mediapipe as mp
import subprocess
import shutil
import tempfile
import pandas as pd
from pathlib import Path
from retinaface import RetinaFace


# 설정
BIN         = "/kaggle/working/OpenFace/build/bin/FeatureExtraction"
OUTPUT_DIR  = Path("/kaggle/working/processed")   # 최종 저장 경로
TARGET_FRAMES = 64      # 영상당 추출 프레임 수
IMG_SIZE      = 256     # 얼굴 이미지 크기
CROP_SCALE    = 2.0     # 1차 크롭 배율 (얼굴 크기의 2배)
CONF_THRESH   = 0.85    # OpenFace confidence 임계값

AU_COLS = [
    "AU01_r","AU02_r","AU04_r","AU05_r","AU06_r","AU07_r",
    "AU09_r","AU10_r","AU12_r","AU14_r","AU15_r","AU17_r",
    "AU20_r","AU23_r","AU24_r","AU25_r","AU26_r",
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [117]:
# 프레임 추출 (Stride Sampling)
def extract_frames(video_path: str, n_frames: int = 64, target_stride: int = 2) -> list[np.ndarray]:
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS)  # 나중에 AU 시간축 계산에 활용 가능

    if total <= 0:
        cap.release()
        return []

    required_total = n_frames * target_stride

    if total <= required_total:
        indices = np.linspace(0, total - 1, n_frames, dtype=int)
    else:
        start   = (total - required_total) // 2
        indices = np.array([start + i * target_stride for i in range(n_frames)])

    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if ret:
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    cap.release()

    # 추출된 프레임이 너무 적으면 None 반환
    # (cap.read 실패로 n_frames보다 적게 추출된 경우 방어)
    if len(frames) < n_frames // 2:
        return []

    return frames

In [89]:
# RetinaFace 대신 insightface 설치
!pip install -q insightface onnxruntime

In [118]:
# detect_face 함수 교체
import insightface
from insightface.app import FaceAnalysis

# 전역 초기화 (한 번만)
FACE_APP = FaceAnalysis(providers=['CPUExecutionProvider'])
FACE_APP.prepare(ctx_id=0, det_size=(640, 640))

def detect_face(frame: np.ndarray) -> dict | None:
    """
    insightface로 얼굴 검출.
    RetinaFace와 동일한 정확도, Keras 충돌 없음.
    """
    faces = FACE_APP.get(frame)
    if not faces:
        return None
    
    # 가장 큰 얼굴 선택
    best = max(faces, key=lambda f: 
               (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]))
    
    x1, y1, x2, y2 = map(int, best.bbox)
    
    # RetinaFace와 동일한 형식으로 반환
    return {
        "facial_area": [x1, y1, x2, y2],
        "landmarks": {
            "left_eye" : best.kps[0].tolist(),
            "right_eye": best.kps[1].tolist(),
        }
    }

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)


In [125]:
# 얼굴 정렬 (MediaPipe 기반)
def align_face(frame: np.ndarray, face_info: dict) -> np.ndarray | None:
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = face_info["facial_area"]
    landmarks = face_info["landmarks"]
    face_w = x2 - x1
    face_h = y2 - y1

    # 1차 크롭 : 얼굴 크기의 CROP_SCALE 배로 넉넉하게
    pad_w = int(face_w * CROP_SCALE)
    pad_h = int(face_h * CROP_SCALE)
    cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
    crop_x1 = max(0, cx - pad_w)
    crop_y1 = max(0, cy - pad_h)
    crop_x2 = min(w, cx + pad_w)
    crop_y2 = min(h, cy + pad_h)

    # 경계를 벗어난 경우 BORDER_CONSTANT(검정)로 패딩
    top    = max(0, -(cy - pad_h))
    bottom = max(0, (cy + pad_h) - h)
    left   = max(0, -(cx - pad_w))
    right  = max(0, (cx + pad_w) - w)

    cropped = frame[crop_y1:crop_y2, crop_x1:crop_x2]
    if top or bottom or left or right:
        cropped = cv2.copyMakeBorder(
            cropped, top, bottom, left, right,
            cv2.BORDER_CONSTANT, value=(0, 0, 0)
        )

    # 눈 좌표 변환 (원본 → 크롭+패딩 좌표계)
    left_eye_x  = landmarks["left_eye"][0]  - crop_x1 + left
    left_eye_y  = landmarks["left_eye"][1]  - crop_y1 + top
    right_eye_x = landmarks["right_eye"][0] - crop_x1 + left
    right_eye_y = landmarks["right_eye"][1] - crop_y1 + top
    left_eye  = np.array([left_eye_x,  left_eye_y],  dtype=np.float32)
    right_eye = np.array([right_eye_x, right_eye_y], dtype=np.float32)

    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dy, dx))

    ch, cw = cropped.shape[:2]
    eye_center = (left_eye + right_eye) / 2
    M = cv2.getRotationMatrix2D(
        (float(eye_center[0]), float(eye_center[1])), angle, scale=1.0
    )
    rotated = cv2.warpAffine(
        cropped, M, (cw, ch),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(0, 0, 0)
    )

    # 2차 크롭
    eye_center_rot = (M @ np.array([*eye_center, 1])).astype(int)
    half = int(max(face_w, face_h) * 0.85)
    fx1 = max(0, eye_center_rot[0] - half)
    fy1 = max(0, eye_center_rot[1] - half)
    fx2 = min(cw, eye_center_rot[0] + half)
    fy2 = min(ch, eye_center_rot[1] + half)

    face_crop = rotated[fy1:fy2, fx1:fx2]
    if face_crop.size == 0:
        return None

    # 256×256 리사이즈 + 검정 패딩
    fh, fw = face_crop.shape[:2]
    scale = IMG_SIZE / max(fh, fw)
    new_w, new_h = int(fw * scale), int(fh * scale)
    resized = cv2.resize(face_crop, (new_w, new_h), interpolation=cv2.INTER_AREA)

    final = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
    y_off = (IMG_SIZE - new_h) // 2
    x_off = (IMG_SIZE - new_w) // 2
    final[y_off:y_off+new_h, x_off:x_off+new_w] = resized

    return final

In [129]:
# OpenFace AU 추출 (정렬된 이미지 폴더 입력)
def extract_au_from_frames(frame_dir: Path, out_dir: Path) -> np.ndarray | None:
    # 1. 이전 결과 삭제 (파일 꼬임 방지)
    if out_dir.exists():
        import shutil
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # 2. OpenFace 실행 (-as_is 추가로 속도 및 정확도 향상)
    cmd = f'"{BIN}" -fdir "{frame_dir}" -out_dir "{out_dir}" -aus -q'
    subprocess.call(cmd, shell=True)

    csv_files = list(out_dir.glob("*.csv"))
    if not csv_files:
        return None

    df = pd.read_csv(csv_files[0])
    df.columns = df.columns.str.strip()

    # 3. 데이터 필터링
    df = df[df["confidence"] >= CONF_THRESH].reset_index(drop=True)
    if len(df) == 0:
        return None

    # 4. 고정된 순서로 AU 추출 (컬럼 누락 방어)
    au_list = []
    for col in AU_COLS:
        if col in df.columns:
            au_list.append(df[col].values)
        else:
            # 해당 AU 컬럼이 없으면 0으로 가득 찬 배열 생성
            au_list.append(np.zeros(len(df), dtype=np.float32))
    
    # 리스트를 쌓아서 (T, 17) 형태로 변환
    au_matrix = np.stack(au_list, axis=1).astype(np.float32)
    # Min-Max (0~5 → 0~1) 정규화
    au_matrix = au_matrix / 5.0
    return au_matrix

In [136]:
# 시퀀스 구성 (64 프레임으로 고정)
def pad_or_crop(arr: np.ndarray, target: int = 64) -> np.ndarray:
    """
    T > target : 앞에서 target개 자르기
    T < target : 마지막 프레임/행으로 반복 패딩
    """
    T = len(arr)
    if T == target:
        return arr
    if T > target:
        return arr[:target]

    last_row = arr[-1:]
    reps = [target - T] + [1] * (arr.ndim - 1)
    pad = np.tile(last_row, reps)
    return np.concatenate([arr, pad], axis=0)

In [141]:
# 단일 영상 전체 처리 함수
def get_aligned_frames(video_path: str, n_frames: int) -> list:
    """stride=2로 시도 후 부족하면 stride=1로 재시도"""
    for stride in [2, 1]:
        frames = extract_frames(video_path, n_frames, target_stride=stride)
        aligned = []
        for frame in frames:
            face_info = detect_face(frame)
            if face_info is None:
                continue
            result = align_face(frame, face_info)
            if result is not None:
                aligned.append(result)
        if len(aligned) >= n_frames // 2:
            return aligned
    return aligned


def process_video(video_path: str, video_id: str, label: int, source: str) -> bool:
    """
    영상 1개 전체 처리.
    저장 구조:
      /kaggle/working/processed/
        {video_id}/
          frames/          ← 정렬된 PNG (64장)
            0000.png ~ 0063.png
          au_sequence.npy  ← shape (64, 17)
          meta.json        ← video_id, label, n_frames 등
    """
    save_dir = OUTPUT_DIR / video_id
    if save_dir.exists():
        if (save_dir / "au_sequence.npy").exists() and (save_dir / "meta.json").exists():
            return True

    save_dir.mkdir(parents=True, exist_ok=True)
    frames_dir = save_dir / "frames"
    frames_dir.mkdir(exist_ok=True)

    # STEP 1~3: 프레임 추출 + 얼굴 검출 + 정렬
    aligned_frames = get_aligned_frames(video_path, TARGET_FRAMES)

    if len(aligned_frames) < 32:
        shutil.rmtree(save_dir)
        return False

    # 64프레임으로 패딩/크롭 후 PNG 저장
    aligned_np = np.array(aligned_frames)
    aligned_np = pad_or_crop(aligned_np, TARGET_FRAMES)

    for i, img in enumerate(aligned_np):
        cv2.imwrite(
            str(frames_dir / f"{i:04d}.png"),
            cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        )

    # STEP 4: OpenFace AU 추출
    au_seq = None
    try:
        with tempfile.TemporaryDirectory(dir="/kaggle/working") as tmpdir:
            au_out    = Path(tmpdir) / "au_out"
            au_matrix = extract_au_from_frames(frames_dir, au_out)

            if au_matrix is None:
                raise ValueError("AU extraction failed")

            au_seq = pad_or_crop(au_matrix, TARGET_FRAMES)
            np.save(save_dir / "au_sequence.npy", au_seq)

    except Exception as e:
        print(f"Error processing {video_id}: {e}")
        if save_dir.exists():
            shutil.rmtree(save_dir)
        return False

    # STEP 5: 메타데이터 저장
    meta = {
        "video_id"         : video_id,
        "label"            : label,
        "source"           : source,
        "n_original_frames": len(aligned_frames),
        "target_frames"    : TARGET_FRAMES,
        "au_shape"         : list(au_seq.shape),
        "video_path"       : str(video_path),
    }
    with open(save_dir / "meta.json", "w") as f:
        json.dump(meta, f, indent=4)

    return True

In [142]:
# ✅ 테스트용으로 임시 교체
success, fail = 0, 0
for i, v in enumerate(tqdm(video_list[:5], desc="테스트 실행")):
    stem     = Path(v["path"]).stem.replace(" ", "_")
    video_id = f"test_{i:03d}_{stem}"
    ok = process_video(v["path"], video_id, v["label"], v["source"])
    print(f"{'✅' if ok else '❌'} {video_id}")
    if ok:
        success += 1
    else:
        fail += 1
print(f"\n결과: 성공 {success} / 실패 {fail}")

'''
테스트 후 교체
# ── 전체 처리 ─────────────────────────────────────────────────────
success, fail = 0, 0
for i, v in enumerate(tqdm(video_list, desc="처리 중")):
    stem     = Path(v["path"]).stem.replace(" ", "_")  # ← 여기
    video_id = f"vid_{i:06d}_{stem}"                   # ← 여기
    ok = process_video(v["path"], video_id, v["label"], v["source"])
    if ok:
        success += 1
    else:
        fail += 1

print(f"\n✅ 완료: 성공 {success} / 실패 {fail}")
'''

테스트 실행:   0%|          | 0/5 [00:00<?, ?it/s]

Could not find the HAAR face detector location
Reading the landmark detector/tracker from: /kaggle/working/OpenFace/build/bin/model/main_ceclm_general.txt
Reading the landmark detector module from: /kaggle/working/OpenFace/build/bin/model/cen_general.txt
Reading the PDM module from: /kaggle/working/OpenFace/build/bin/model/pdms/In-the-wild_aligned_PDM_68.txt....Done
Reading the Triangulations module from: /kaggle/working/OpenFace/build/bin/model/tris_68.txt....Done
Reading the intensity CEN patch experts from: /kaggle/working/OpenFace/build/bin/model/patch_experts/cen_patches_0.25_of.dat....Done
Reading the intensity CEN patch experts from: /kaggle/working/OpenFace/build/bin/model/patch_experts/cen_patches_0.35_of.dat....Done
Reading the intensity CEN patch experts from: /kaggle/working/OpenFace/build/bin/model/patch_experts/cen_patches_0.50_of.dat....Done
Reading the intensity CEN patch experts from: /kaggle/working/OpenFace/build/bin/model/patch_experts/cen_patches_1.00_of.dat....Don

테스트 실행:  20%|██        | 1/5 [00:42<02:49, 42.29s/it]

0% 10% 20% 30% 40% 50% 60% 70% 80% 90% 100% 
Closing output recorder
Closing input reader
Closed successfully
Postprocessing the Action Unit predictions
✅ test_000_26__walking_down_street_outside_angry
Could not find the HAAR face detector location
Reading the landmark detector/tracker from: /kaggle/working/OpenFace/build/bin/model/main_ceclm_general.txt
Reading the landmark detector module from: /kaggle/working/OpenFace/build/bin/model/cen_general.txt
Reading the PDM module from: /kaggle/working/OpenFace/build/bin/model/pdms/In-the-wild_aligned_PDM_68.txt....Done
Reading the Triangulations module from: /kaggle/working/OpenFace/build/bin/model/tris_68.txt....Done
Reading the intensity CEN patch experts from: /kaggle/working/OpenFace/build/bin/model/patch_experts/cen_patches_0.25_of.dat....Done
Reading the intensity CEN patch experts from: /kaggle/working/OpenFace/build/bin/model/patch_experts/cen_patches_0.35_of.dat....Done
Reading the intensity CEN patch experts from: /kaggle/working/

테스트 실행:  40%|████      | 2/5 [01:24<02:07, 42.41s/it]

...

테스트 실행:  80%|████████  | 4/5 [03:17<00:50, 50.17s/it]

0% 10% 20% 30% 40% 50% 60% 70% 80% 90% 100% 
Closing output recorder
Closing input reader
Closed successfully
Postprocessing the Action Unit predictions
✅ test_003_08__walking_down_street_outside_angry
Could not find the HAAR face detector location
Reading the landmark detector/tracker from: /kaggle/working/OpenFace/build/bin/model/main_ceclm_general.txt
Reading the landmark detector module from: /kaggle/working/OpenFace/build/bin/model/cen_general.txt
Reading the PDM module from: /kaggle/working/OpenFace/build/bin/model/pdms/In-the-wild_aligned_PDM_68.txt....Done
Reading the Triangulations module from: /kaggle/working/OpenFace/build/bin/model/tris_68.txt....Done
Reading the intensity CEN patch experts from: /kaggle/working/OpenFace/build/bin/model/patch_experts/cen_patches_0.25_of.dat....Done
Reading the intensity CEN patch experts from: /kaggle/working/OpenFace/build/bin/model/patch_experts/cen_patches_0.35_of.dat....Done
Reading the intensity CEN patch experts from: /kaggle/working/

테스트 실행: 100%|██████████| 5/5 [04:05<00:00, 49.19s/it]

0% 10% 20% 30% 40% 50% 60% 70% 80% 90% 100% 
Closing output recorder
Closing input reader
Closed successfully
Postprocessing the Action Unit predictions
✅ test_004_05__outside_talking_still_laughing

결과: 성공 5 / 실패 0


'\n테스트 후 교체\n# ── 전체 처리 ─────────────────────────────────────────────────────\nsuccess, fail = 0, 0\nfor i, v in enumerate(tqdm(video_list, desc="처리 중")):\n    stem     = Path(v["path"]).stem.replace(" ", "_")  # ← 여기\n    video_id = f"vid_{i:06d}_{stem}"                   # ← 여기\n    ok = process_video(v["path"], video_id, v["label"], v["source"])\n    if ok:\n        success += 1\n    else:\n        fail += 1\n\nprint(f"\n✅ 완료: 성공 {success} / 실패 {fail}")\n'

In [78]:
from pathlib import Path
import numpy as np

OUTPUT_DIR = Path("/kaggle/working/processed")

for d in OUTPUT_DIR.glob("test_*"):
    print(f"\n{d.name}")
    
    # 파일 목록
    for f in d.rglob("*"):
        if f.is_file():
            print(f"  {f.name}")
    
    # AU 시퀀스 확인
    au_path = d / "au_sequence.npy"
    if au_path.exists():
        au = np.load(au_path)
        print(f"  AU shape: {au.shape}")  # (64, 17) 이어야 함
        print(f"  AU 샘플값: {au[0]}")


test_004_05__outside_talking_still_laughing
  meta.json
  au_sequence.npy
  0043.png
  0019.png
  0021.png
  0052.png
  0009.png
  0055.png
  0045.png
  0020.png
  0010.png
  0004.png
  0047.png
  0016.png
  0030.png
  0038.png
  0014.png
  0033.png
  0018.png
  0054.png
  0000.png
  0056.png
  0011.png
  0028.png
  0050.png
  0061.png
  0024.png
  0032.png
  0040.png
  0001.png
  0053.png
  0048.png
  0022.png
  0003.png
  0034.png
  0027.png
  0044.png
  0008.png
  0012.png
  0007.png
  0041.png
  0049.png
  0039.png
  0025.png
  0036.png
  0013.png
  0046.png
  0002.png
  0017.png
  0037.png
  0005.png
  0023.png
  0031.png
  0060.png
  0006.png
  0035.png
  0042.png
  0026.png
  0029.png
  0063.png
  0059.png
  0015.png
  0062.png
  0057.png
  0058.png
  0051.png
  AU shape: (64, 17)
  AU 샘플값: [0.188      0.         0.34600002 0.         0.252      0.378
 0.         0.352      0.174      0.21       0.         0.094
 0.         0.         0.         0.16       0.094     ]

test_003

In [ ]:
# Train / Val / Test 분할
import re
import json
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

# 처리된 영상 메타데이터 수집
records = []
for meta_file in OUTPUT_DIR.rglob("meta.json"):
    with open(meta_file) as f:
        records.append(json.load(f))

df = pd.DataFrame(records)

# Celeb-DF 테스트 리스트를 미리 메모리에 로드
CELEB_TEST_SET = set()
test_list_path = CELEB_ROOT / "List_of_testing_videos.txt"
if test_list_path.exists():
    with open(test_list_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                CELEB_TEST_SET.add(parts[1])

def get_split(video_path: str, source: str) -> str:
    path = Path(video_path)

    if source == "DFD":
        m = re.match(r"^(\d{2})_\d{2}__", path.stem)
        if m:
            target_num = int(m.group(1))
            if target_num <= 80:
                return "train"
            elif target_num <= 90:
                return "val"
            else:
                return "test"
        return "train"

    elif source == "CelebDF":
        rel_path = f"{path.parent.name}/{path.name}"
        if rel_path in CELEB_TEST_SET:
            return "test"
        return "train"  # 일단 train으로 설정 (아래에서 일부를 val로 변경)

    return "train"

df["split"] = df.apply(
    lambda row: get_split(row["video_path"], row["source"]), axis=1
)

# CelebDF train 중 10%를 val로 랜덤 할당
# stratify=label로 real/fake 비율 유지
celeb_train_idx = df[
    (df["source"] == "CelebDF") & (df["split"] == "train")
].index

if len(celeb_train_idx) > 0:
    _, val_idx = train_test_split(
        celeb_train_idx,
        test_size=0.1,
        random_state=42,
        stratify=df.loc[celeb_train_idx, "label"]
    )
    df.loc[val_idx, "split"] = "val"

# 저장
df.to_csv(OUTPUT_DIR / "dataset_metadata.csv", index=False)

print("=== 분할 결과 ===")
for source in ["DFD", "CelebDF", "전체"]:
    print(f"\n[{source}]")
    sub = df if source == "전체" else df[df["source"] == source]
    for split in ["train", "val", "test"]:
        s = sub[sub["split"] == split]
        if len(s) == 0:
            continue
        print(f"  {split:5}: {len(s):4}개 "
              f"(real={(s.label==0).sum()}, fake={(s.label==1).sum()})")

In [ ]:
# PyTorch Dataset 클래스
import torch
from torch.utils.data import Dataset

class DeepfakeDataset(Dataset):
    """
    이미지 시퀀스 (64, 256, 256, 3) + AU 시퀀스 (64, 17) 동시 반환.
    Late Fusion 모델 입력용.
    """
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        save_dir = OUTPUT_DIR / row["video_id"]

        try:
            # 이미지 시퀀스 로딩 (T, H, W, C) → (T, C, H, W)
            frames_dir = save_dir / "frames"
            imgs = []
            for png in sorted(frames_dir.glob("*.png"))[:64]:
                img = cv2.imread(str(png))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                imgs.append(img)

            # transform이 있으면 64장 전체에 동일한 변환 적용
            # 핵심: seed를 고정해서 모든 프레임에 같은 랜덤 변환 보장
            if self.transform:
                seed = np.random.randint(0, 2**31)
                transformed = []
                for img in imgs:
                    np.random.seed(seed)
                    torch.manual_seed(seed)
                    transformed.append(self.transform(img))
                img_tensor = torch.stack(transformed)  # (T, C, H, W)
            else:
                img_seq    = np.stack(imgs, axis=0).transpose(0, 3, 1, 2)
                img_tensor = torch.from_numpy(img_seq).float() / 255.0

            # AU 시퀀스 로딩 (64, 17)
            au_seq    = np.load(save_dir / "au_sequence.npy")
            au_tensor = torch.from_numpy(au_seq).float()

            label = torch.tensor(row["label"], dtype=torch.long)
            return img_tensor, au_tensor, label

        except Exception as e:
            print(f"Error loading {row['video_id']}: {e}")
            for _ in range(3):
                new_idx = np.random.randint(0, len(self.df))
                if new_idx != idx:
                    return self.__getitem__(new_idx)
            raise RuntimeError(f"Failed to load sample after retries: {row['video_id']}")

In [146]:
import numpy as np

# 처리된 첫 번째 영상의 AU 파일 경로
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/processed")

# 첫 번째 test_ 폴더 찾기
au_files = list(OUTPUT_DIR.rglob("au_sequence.npy"))
print(f"AU 파일 수: {len(au_files)}")

if au_files:
    au = np.load(au_files[0])
    print(f"경로: {au_files[0]}")
    print(f"shape: {au.shape}")
    print(au)

AU 파일 수: 5
경로: /kaggle/working/processed/test_004_05__outside_talking_still_laughing/au_sequence.npy
shape: (64, 17)
[[0.184      0.         0.292      ... 0.         0.184      0.344     ]
 [0.136      0.012      0.28399998 ... 0.         0.148      0.268     ]
 [0.08       0.012      0.258      ... 0.         0.108      0.206     ]
 ...
 [0.10599999 0.012      0.328      ... 0.         0.21199998 0.296     ]
 [0.048      0.022      0.282      ... 0.         0.094      0.136     ]
 [0.         0.048      0.23599999 ... 0.         0.         0.        ]]
